# Data Exploration

This notebook explores the RadioML datasets used for modulation classification.

**Contents:**
- Load and inspect RadioML2016.10a dataset
- Visualize signal constellations
- Analyze class distribution and SNR ranges

In [ ]:
import sys
from pathlib import Path

# Add src to path if running from notebooks directory
src_path = Path("../src")
if src_path.exists():
    sys.path.insert(0, str(src_path.resolve()))

import numpy as np
import matplotlib.pyplot as plt

from robust_amc.data import (
    load_radioml2016a,
    stratified_split,
    MODULATION_CLASSES,
    SNR_LEVELS,
)
from robust_amc.evaluation import plot_constellation, plot_constellation_grid

## 1. Load Dataset

In [ ]:
DATA_PATH = Path("../data/RML2016.10a_dict.pkl")

if not DATA_PATH.exists():
    print(f"Dataset not found at {DATA_PATH}")
    print("Please download RadioML2016.10a from https://www.deepsig.ai/datasets/")
else:
    data, labels, snrs = load_radioml2016a(DATA_PATH)
    print(f"Dataset loaded successfully!")
    print(f"  Shape: {data.shape}")
    print(f"  Classes: {len(MODULATION_CLASSES)}")
    print(f"  SNR range: {snrs.min()} to {snrs.max()} dB")

## 2. Class Distribution

In [ ]:
unique, counts = np.unique(labels, return_counts=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([MODULATION_CLASSES[i] for i in unique], counts)
ax.set_xlabel("Modulation Type")
ax.set_ylabel("Number of Samples")
ax.set_title("Class Distribution")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("\nSamples per class:")
for idx, count in zip(unique, counts):
    print(f"  {MODULATION_CLASSES[idx]}: {count}")

## 3. Signal Constellations

Visualize I/Q constellations for different modulation types at various SNR levels.

In [ ]:
# Select a few modulations to visualize
selected_mods = ["BPSK", "QPSK", "8PSK", "QAM16", "QAM64"]
snr_to_plot = 18  # High SNR for clearer constellations

fig, axes = plt.subplots(1, len(selected_mods), figsize=(15, 3))

for ax, mod_name in zip(axes, selected_mods):
    if mod_name in MODULATION_CLASSES:
        mod_idx = MODULATION_CLASSES.index(mod_name)
        mask = (labels == mod_idx) & (snrs == snr_to_plot)
        if mask.sum() > 0:
            sample = data[mask][0]
            plot_constellation(sample, title=mod_name, ax=ax)
        else:
            ax.set_title(f"{mod_name} (no data)")
    else:
        ax.set_title(f"{mod_name} (not found)")

plt.suptitle(f"Signal Constellations at SNR = {snr_to_plot} dB")
plt.tight_layout()
plt.show()

## 4. SNR Impact on Constellations

See how noise affects the constellation clarity.

In [ ]:
# Plot QPSK at different SNR levels
mod_name = "QPSK"
mod_idx = MODULATION_CLASSES.index(mod_name)
snr_levels_to_plot = [-10, 0, 10, 18]

fig, axes = plt.subplots(1, len(snr_levels_to_plot), figsize=(12, 3))

for ax, snr in zip(axes, snr_levels_to_plot):
    mask = (labels == mod_idx) & (snrs == snr)
    if mask.sum() > 0:
        sample = data[mask][0]
        plot_constellation(sample, title=f"{mod_name} @ {snr} dB", ax=ax)

plt.suptitle(f"{mod_name} Constellation vs SNR")
plt.tight_layout()
plt.show()

## 5. Train/Val/Test Split

In [ ]:
splits = stratified_split(data, labels, snrs)

print("Dataset splits:")
for split_name, (split_data, split_labels, split_snrs) in splits.items():
    print(f"  {split_name}: {len(split_labels)} samples")